In [ ]:
from datasets import Dataset, Features, Image, Value
from qwen_vl_utils import process_vision_info
from pathlib import Path
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, Qwen2VLForConditionalGeneration, AutoProcessor,AutoTokenizer

from modelscope import snapshot_download, AutoTokenizer
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq, Qwen2VLForConditionalGeneration, AutoProcessor
import torch

from swanlab.integration.transformers import SwanLabCallback
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from functools import partial

from functools import partial

import torch
import pandas as pd
import random
import swanlab
import json
import os
# 配置镜像源和加速参数
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"    # 镜像源
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"          # 启用高效传输协议
os.environ["HF_HOME"] = "./"             # 指定统一缓存目录


In [ ]:
from tools import process, model_download_localload, model_download_mirror, data_load_process
from exp import train
#参数存储
from config_2b_formula import *  


In [ ]:
if Path(model_base_dir).exists():
    print("检测到本地模型，尝试加载...")
    model, tokenizer, processor = model_download_localload.load_model_from_local(model_base_dir)
else:
    if model_base_dir=="Salesforce/blip2-flan-t5-xl":
        print("本地模型不存在，正在下载...")
        model, tokenizer, processor = model_download_mirror.model_download_mirror(model_name)
    elif model_base_dir=="Qwen/Qwen2-VL-2B-Instruct":
        model, tokenizer, processor = model_download_mirror.model_download(model_name)
    elif model_base_dir=="Qwen/Qwen2.5-VL-7B-Instruct":
        model, tokenizer, processor = model_download_mirror.model_download(model_name)

In [ ]:
data_load_process.conversation_generator(csv_path,train_json_path,test_json_path,val_json_path)

In [ ]:
train_ds = Dataset.from_json(train_dataset_json_path)

In [ ]:
bound_process_func = partial(process.process_func, processor=processor, tokenizer=tokenizer,prompt=prompt)
# 应用处理函数到数据集
train_dataset = train_ds.map(bound_process_func)

In [ ]:
swanlab_callback1 = SwanLabCallback(
    project="MutiDataTrained-latexocr1",
    experiment_name="2B-formula-4k-data",
    config={
        "model": "https://modelscope.cn/models/Qwen/Qwen2-VL-7B-Instruct",
        "dataset":" https://huggingface.co/datasets/HuggingFaceM4/datikz",
        # "github": "https://github.com/datawhalechina/self-llm",
        "model_id": model_name,
        "train_dataset_json_path": train_json_path,
        "val_dataset_json_path": val_json_path,
        "output_dir": output_dir,
        "prompt": prompt,
        "train_data_number": len(train_ds),
        "token_max_length":8096,
        "lora_rank": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
    },
)


In [ ]:
def model_train_func(model,output_dir,train_dataset,swanlab_callback,tokenizer):
    
    config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"，],
    inference_mode=False,  # 训练模式
    r=16,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
    bias="none",
)

# 获取LoRA模型
    train_peft_model = get_peft_model(model, config)
    
    # 配置训练参数
    args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    logging_steps=1,
    logging_first_step=1,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
)


    trainer = Trainer(
    model=train_peft_model,
    args=args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[swanlab_callback],
)

    
    return trainer


In [ ]:
trainer=model_train_func(model,output_dir,train_dataset,swanlab_callback1,tokenizer)

In [ ]:
trainer.train()

In [ ]:
swanlab.finish()

In [ ]:
def predict(messages, model):
    # 准备推理
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")

    # 生成输出
    generated_ids = model.generate(**inputs, max_new_tokens=8196)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    
    return output_text[0]


In [ ]:
val_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
    inference_mode=True,  # 训练模式
    r=16,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.05,  # Dropout 比例
    bias="none",
)

# 获取测试模型，从output_dir中获取最新的checkpoint
load_model_path = f"{output_dir}/checkpoint-{max([int(d.split('-')[-1]) for d in os.listdir(output_dir) if d.startswith('checkpoint-')])}"
print(f"load_model_path: {load_model_path}")
val_peft_model = PeftModel.from_pretrained(model, model_id=load_model_path, config=val_config)

# 读取测试数据
with open(val_json_path, "r") as f:
    test_dataset = json.load(f)



In [ ]:
test_image_list = []
for item in test_dataset:
    image_file_path = item["conversations"][0]["value"]
    label = item["conversations"][1]["value"]
    
    messages = [{
        "role": "user", 
        "content": [
            {
            "type": "image", 
            "image": image_file_path,
            "resized_height": 224,
            "resized_width": 224,   
            },
            {
            "type": "text",
            "text": prompt,
            }
        ]}]

    response = predict(messages, val_peft_model)
    
    print(f"predict:{response}")
    print(f"gt:{label}\n")

    test_image_list.append(swanlab.Image(image_file_path, caption=response))

swanlab.log({"Prediction": test_image_list})

# 在Jupyter Notebook中运行时要停止SwanLab记录，需要调用swanlab.finish()
swanlab.finish()


In [ ]:
table_train_ds = Dataset.from_json("table_latex_ocr_train.json")

In [ ]:
bound_process_func = partial(process.process_func, processor=processor, tokenizer=tokenizer,prompt=prompt)
# 应用处理函数到数据集
table_train_dataset = table_train_ds.map(bound_process_func)

In [ ]:
val_peft_model = PeftModel.from_pretrained(model, model_id=load_model_path)

In [ ]:
swanlab_callback1 = SwanLabCallback(
    project="MutiDataTrained-latexocr1-step2",
    experiment_name="2B-formula-4k-data",
    config={
        "model": "https://modelscope.cn/models/Qwen/Qwen2-VL-7B-Instruct",
        "dataset":" https://huggingface.co/datasets/HuggingFaceM4/datikz",
        # "github": "https://github.com/datawhalechina/self-llm",
        "model_id": model_name,
        "train_dataset_json_path": train_json_path,
        "val_dataset_json_path": val_json_path,
        "output_dir": output_dir,
        "prompt": prompt,
        "train_data_number": len(train_ds),
        "token_max_length":8096,
        "lora_rank": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
    },
)

In [ ]:
trainer=model_train_func(val_peft_model ,output_dir,table_train_dataset,swanlab_callback1,tokenizer)

In [ ]:
trainer.train()